In [1]:
from __future__ import annotations

import os
import sys

# Make current directory available to the notebook
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# Change working directory to the notebook's directory if needed
notebook_dir = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in globals()
    else os.getcwd()
)
os.chdir(notebook_dir)

In [ ]:
import json
from dataclasses import dataclass

from pydantic_ai import Agent  # core LLM wrapper
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider
from pydantic_graph import BaseNode, End, Graph, GraphRunContext

# ---------------------------------------------------------------------------
# 1.  Build the SUB‑GRAPH ----------------------------------------------------
# ---------------------------------------------------------------------------
#   It consists of a single node which calls an LLM to calculate the length
#   of the input text and returns an ``int`` via ``End[int]``.
# ---------------------------------------------------------------------------

# a very small agent that just returns ``len(text)`` - replace the model name
# with whatever backend you have configured ("openai:gpt-4o" is used here as
# it is short and available in the docs).


with open("../env.json", "r", encoding="utf-8") as f:
    config: dict[str, str] = json.load(f)
provider = GoogleProvider(api_key=config["GOOGLE_API_KEY"])
model = GoogleModel("gemini-1.5-flash", provider=provider)

length_agent: Agent[None, int] = Agent(
    model,
    output_type=int,
    system_prompt="You are a strict function that ONLY returns the number of characters you receive as input.",
)


@dataclass
class TextLength(BaseNode[None, None, int]):
    """Node that ends the *sub_graph* by returning `End[int]`."""

    text: str

    async def run(self, ctx: GraphRunContext[None]) -> End[int]:
        """Call the LLM agent and end the graph with its numeric result."""
        result = await length_agent.run(self.text)
        return End(result.output)


# the sub‑graph itself - could contain many nodes; one is enough for the demo
sub_graph: Graph[None, None, int] = Graph(nodes=[TextLength])

In [2]:
# ---------------------------------------------------------------------------
# 2.  Build the MAIN GRAPH ---------------------------------------------------
# ---------------------------------------------------------------------------
#   The main graph has two nodes:
#       • ``RunSubGraph`` - kicks off *sub_graph* and receives its integer
#         output.
#       • ``HandleResult`` - does some arbitrary follow‑up work and ends the
#         main graph.
# ---------------------------------------------------------------------------

CONST_TO_ADD = 42  # prove that the main graph keeps running after sub_graph


@dataclass
class RunSubGraph(BaseNode):
    """Entry node of *main_graph* - runs the sub_graph and passes its result."""

    text: str

    async def run(self, ctx: GraphRunContext[None]) -> "HandleResult":
        # Call the sub graph **inside** the node - it is just a function call.
        # Note: because we are *already* inside an async context, we `await` the
        # sub graph directly.
        sub_result = await sub_graph.run(TextLength(self.text))
        # Pass the numeric output on to the next node.
        return HandleResult(sub_result.output)


@dataclass
class HandleResult(BaseNode[None, None, int]):
    """Continue after the sub_graph; eventually ends the main graph."""

    length: int

    async def run(self, ctx: GraphRunContext[None]) -> End[int]:
        # Do something with the sub‑graph output. Here we just add a constant.
        final_value = self.length + CONST_TO_ADD
        return End(final_value)


# Compose the main graph - order of nodes does not matter
main_graph: Graph[None, None, int] = Graph(nodes=[RunSubGraph, HandleResult])


# ---------------------------------------------------------------------------
# 3.  Demo runner ------------------------------------------------------------
# ---------------------------------------------------------------------------
#   Everything above defines the graphs. The following `main()` function is a
#   minimal runnable example so you can see the interaction in action.
# ---------------------------------------------------------------------------


async def main() -> None:  # pragma: no cover
    """Run the main graph synchronously for demonstration purposes."""
    user_text = "Pydantic graphs make orchestrating async workflows elegant!"
    result = await main_graph.run(RunSubGraph(user_text))
    print(
        f"Input text length   : {len(user_text)}",  # ground‑truth check
        # f"Sub‑graph output    : {result.history[1].state if result.history else 'n/a'}",  # optional
        f"Main graph final val: {result.output}",
        sep="\n",
    )


# Run the main function directly with await instead of asyncio.run()
await main()

Input text length   : 59
Main graph final val: 79
